# LLM-as-a-Judge: Synthetic Data Validation (Gemini Flash)

Validates the synthetic hate speech dataset generated for 20 Singlish terms.

**How it works:**
1. Load ground truth term definitions from `chosen-20-terms.json`
2. Load synthetic examples from train/val/test splits
3. For each example, ask Gemini Flash to **blindly** classify it (without seeing the assigned label)
4. Compare judge labels vs assigned labels → agreement metrics, confusion matrix
5. Detect near-duplicate texts, assess Singlish realism
6. Export summary CSV of all judge results

**Configuration:** Set `GEMINI_KEY` in a `.env` file at the repo root.

## Configure Gemini API
Loads `GEMINI_KEY` from `.env` file at repo root.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Load .env from repo root (two levels up from notebooks/synthetic/)
REPO_ROOT = Path("../..").resolve()
load_dotenv(REPO_ROOT / ".env")

GEMINI_KEY = os.getenv("GEMINI_KEY")
GEMINI_MODEL = "gemini-3-flash-preview"

if not GEMINI_KEY:
    raise ValueError("GEMINI_KEY not found. Add it to .env at the repo root.")

GENAI_CLIENT = genai.Client(api_key=GEMINI_KEY)

print("✅ Gemini configured")
print("✅ Model:", GEMINI_MODEL)

✅ Gemini configured
✅ Model: gemini-3-flash-preview


## Load ground truth and data

In [2]:
import json
import re
import time
import csv
import difflib
from collections import Counter, defaultdict

# ---------- paths ----------
DATA_DIR = REPO_ROOT / "data" / "singlish" / "synthetic" / "20-terms"
GROUND_TRUTH_FILE = DATA_DIR / "chosen-20-terms.json"
JUDGE_OUTPUT_FILE = DATA_DIR / "judge_results.jsonl"
JUDGE_CSV_FILE = DATA_DIR / "judge_results.csv"

# Which splits to judge (change to e.g. ['test'] to judge only one)
SPLITS_TO_JUDGE = ["train", "val", "test"]

# ---------- load ground truth ----------
with open(GROUND_TRUTH_FILE, encoding="utf-8") as f:
    ground_truth_list = json.load(f)

TERM_LOOKUP = {entry["term"].casefold(): entry for entry in ground_truth_list}

# ---------- SG target groups (matches generation notebook) ----------
SG_TARGET_GROUPS = {
    "chinese": "Chinese Singaporeans",
    "malay": "Malay Singaporeans",
    "indian": "Indian Singaporeans",
    "eurasian": "Eurasian Singaporeans",
    "tamil": "Tamil community in Singapore",
    "hindi": "Hindi-speaking community in Singapore",
    "punjabi": "Punjabi / Sikh community in Singapore",
    "migrant": "Migrant workers in Singapore (general)",
    "worker": "Foreign workers in Singapore (general)",
    "bangladeshi": "Bangladeshi workers in Singapore",
    "bangla": "Bangladeshi community (colloquial term)",
    "filipino": "Filipino workers / community in Singapore",
    "vietnamese": "Vietnamese workers / community in Singapore",
    "myanmar": "Myanmar workers / community in Singapore",
    "burmese": "Burmese workers / community in Singapore",
    "prc": "Chinese nationals / PRCs in Singapore",
    "muslim": "Muslim community in Singapore",
    "islam": "Islamic community in Singapore",
    "christian": "Christian community in Singapore",
    "buddhist": "Buddhist community in Singapore",
    "hindu": "Hindu community in Singapore",
    "sikh": "Sikh community in Singapore",
    "taoist": "Taoist community in Singapore",
    "foreigner": "Foreigners in Singapore (general)",
    "foreign_worker": "Foreign workers in Singapore",
}

TARGET_GROUP_LIST = "\n".join(f"- {k}: {v}" for k, v in SG_TARGET_GROUPS.items())

def extract_context(entry):
    """Format term context for the judge prompt."""
    definitions = (entry.get("definitions") or "").strip()
    usage_notes = (entry.get("usage_notes") or "").strip()
    categories = entry.get("categories") or []
    parts = []
    if categories:
        parts.append(f"Categories: {', '.join(categories)}")
    if definitions:
        parts.append(f"Definitions:\n{definitions}")
    if usage_notes:
        parts.append(f"Usage notes:\n{usage_notes}")
    text = "\n\n".join(parts).strip()
    return text[:1500] + ("\n..." if len(text) > 1500 else "")

# ---------- load synthetic data ----------
all_rows = []
for split_name in SPLITS_TO_JUDGE:
    split_file = DATA_DIR / f"{split_name}.jsonl"
    if not split_file.exists():
        print(f"⚠ {split_file} not found, skipping")
        continue
    with open(split_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["split"] = split_name
            all_rows.append(row)

print(f"✅ Ground truth: {len(TERM_LOOKUP)} terms loaded")
print(f"✅ Data loaded: {len(all_rows)} rows from splits {SPLITS_TO_JUDGE}")
print(f"   Labels: {Counter(r['label'] for r in all_rows)}")
print(f"   Per split: {Counter(r['split'] for r in all_rows)}")
print(f"   Terms: {sorted(set(r['source_term'] for r in all_rows))}")

✅ Ground truth: 20 terms loaded
✅ Data loaded: 540 rows from splits ['train', 'val', 'test']
   Labels: Counter({'hate': 180, 'normal': 180, 'offensive': 180})
   Per split: Counter({'train': 180, 'val': 180, 'test': 180})
   Terms: ['CB kia', 'Zhu Bajie', 'ah kua', 'anneh', 'apu neh neh', 'big fuck', 'blur cock', 'bo lan pa', 'cheeko pek', 'cina babi', 'fatty bom bom', 'goondu', 'jiak sai', 'keling', 'lan jiao bin', 'nothing but a prostitute', 'pukimak', 'rabak', 'siao cha bor', 'siao lang']


## Judge prompt and API logic

The judge **does not see** the assigned label — it performs a blind independent assessment. It returns:
- `judge_label`: hate / offensive / normal
- `judge_target_group`: which identity group (or "none")
- `judge_confidence`: high / medium / low
- `singlish_realism`: high / medium / low
- `reasoning`: 1-2 sentence explanation

In [4]:
JUDGE_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "judge_label": {"type": "string", "enum": ["hate", "offensive", "normal"]},
        "judge_target_group": {"type": "string"},
        "judge_confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "singlish_realism": {"type": "string", "enum": ["high", "medium", "low"]},
        "reasoning": {"type": "string"},
    },
    "required": ["judge_label", "judge_target_group", "judge_confidence", "singlish_realism", "reasoning"],
}

TEMPERATURE = 0.1  # Low temp for consistent judging


def build_judge_prompt(text, source_term, term_context):
    """Build the blind judge prompt — does NOT reveal the assigned label."""
    return f"""You are an expert judge evaluating a single text sample from a Singapore hate speech dataset.

## Term being used
Term: "{source_term}"
Context about this term:
{term_context}

## Singapore target identity groups
{TARGET_GROUP_LIST}

## Classification rubric
- **hate**: Expresses hatred, dehumanization, calls for violence or exclusion against a specific identity group. Must target a protected group.
- **offensive**: Uses crude, vulgar, or insulting language but is NOT directed at a protected identity group. May insult individuals without targeting their group identity.
- **normal**: Neutral, educational, or descriptive use of the term. No hatred or strong offence intended.

## Text to judge
"{text}"

## Instructions
1. Classify the text as "hate", "offensive", or "normal" based on the rubric above.
2. If the text targets a specific identity group, specify which one from the list above. Otherwise use "none".
3. Rate your confidence in the classification: "high", "medium", or "low".
4. Rate how realistic the text sounds as authentic Singlish or Singapore English: "high" (natural, could be a real post), "medium" (understandable but somewhat artificial), "low" (clearly synthetic/unnatural).
5. Provide a 1-2 sentence reasoning for your classification.

Return ONLY a JSON object with these fields: judge_label, judge_target_group, judge_confidence, singlish_realism, reasoning."""


def _call_gemini_json(prompt, temperature=TEMPERATURE, max_retries=5):
    """Call Gemini with structured JSON output and exponential backoff."""
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            response = GENAI_CLIENT.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config={
                    "temperature": temperature,
                    "top_p": 0.9,
                    "response_mime_type": "application/json",
                    "response_json_schema": JUDGE_RESPONSE_SCHEMA,
                },
            )
            raw = (getattr(response, "text", None) or "").strip()
            return raw
        except Exception as e:
            last_exc = e
            err_str = str(e)
            is_transient = any(
                k in err_str
                for k in ["503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "overloaded", "rate limit"]
            )
            if is_transient and attempt < max_retries:
                wait = min(2**attempt, 120)
                print(f"\n  ⚠ API error (attempt {attempt}/{max_retries}): {err_str[:120]}")
                print(f"  ⏳ Retrying in {wait}s...", flush=True)
                time.sleep(wait)
            else:
                print(f"\n  ❌ API error (attempt {attempt}/{max_retries}): {err_str[:200]}")
                return None
    return None


def parse_judge_response(raw):
    """Parse the judge JSON response."""
    if not raw:
        return None
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None

    if isinstance(data, list):
        data = data[0] if data else None
    if not isinstance(data, dict):
        return None

    # Normalize fields
    data["judge_label"] = str(data.get("judge_label", "")).strip().lower()
    data["judge_target_group"] = str(data.get("judge_target_group", "none")).strip().lower()
    data["judge_confidence"] = str(data.get("judge_confidence", "")).strip().lower()
    data["singlish_realism"] = str(data.get("singlish_realism", "")).strip().lower()
    data["reasoning"] = str(data.get("reasoning", "")).strip()

    if data["judge_label"] not in ("hate", "offensive", "normal"):
        return None
    return data


print(f"✅ Judge prompt builder ready")
print(f"   Schema fields: {list(JUDGE_RESPONSE_SCHEMA['properties'].keys())}")
print(f"   Temperature: {TEMPERATURE}")

✅ Judge prompt builder ready
   Schema fields: ['judge_label', 'judge_target_group', 'judge_confidence', 'singlish_realism', 'reasoning']
   Temperature: 0.1


## Preview judge prompt
Inspect the prompt for one example before running the full pipeline.

In [5]:
# Preview: pick one example from each label
previews = {}
for row in all_rows:
    label = row["label"]
    if label not in previews:
        previews[label] = row
    if len(previews) == 3:
        break

for label, row in previews.items():
    term = row["source_term"]
    entry = TERM_LOOKUP.get(term.casefold(), {})
    context = extract_context(entry)
    prompt = build_judge_prompt(row["text"], term, context)
    print("=" * 100)
    print(f"PREVIEW — assigned_label={label} | term={term}")
    print("=" * 100)
    print(prompt[:800])
    print("..." if len(prompt) > 800 else "")
    print()

PREVIEW — assigned_label=hate | term=ah kua
You are an expert judge evaluating a single text sample from a Singapore hate speech dataset.

## Term being used
Term: "ah kua"
Context about this term:
Categories: insults

Definitions:
1.   (derogatory) an effeminate man or boy

Boys and girls in non-coed schools tend to become ah kua and butch from secondary onwards.

— 2012 December 11, sinoboy. Kiasu Parents, "PSLE 2012 - Results Discussion".

2.   (derogatory) a male transvestite; a crossdresser

Guys in skirts = not guys anymore. Just Ah Kua

— 2019 March 23, FattChoy. Blowing Wind, "A short skirt is freedom… no crotch binding".

3.   (derogatory) a trans woman; a male-to-female transgender person

4.   (derogatory) a gay man

If the same scenario takes place (whereby a gay peep at a straight, wink & lick his lips), I think he will
...

PREVIEW — assigned_label=normal | term=ah kua
You are an expert judge evaluating a single text sample from a Singapore hate speech dataset.

## Term b

## Browse dataset
View all loaded rows with their index, split, label, target group, and text.

In [6]:
# Print all rows with index for easy reference
print(f"{'idx':>4}  {'split':>6}  {'label':>10}  {'target_group':>15}  {'term':>25}  text")
print("-" * 120)
for idx, row in enumerate(all_rows):
    text_preview = row["text"][:60].replace("\n", " ")
    print(
        f"{idx:>4}  {row['split']:>6}  {row['label']:>10}  "
        f"{row.get('target_group', 'none'):>15}  {row['source_term']:>25}  {text_preview}"
    )
print(f"\nTotal: {len(all_rows)} rows")

 idx   split       label     target_group                       term  text
------------------------------------------------------------------------------------------------------------------------
   0   train        hate          chinese                     ah kua  Chinese Singaporeans who are ah kua are a complete embarrass
   1   train        hate            malay                     ah kua  Malay Singaporeans who act like that are nothing but a bunch
   2   train        hate           indian                     ah kua  Indian Singaporeans are nothing but a bunch of weak ah kua w
   3   train      normal             none                     ah kua  The term ah kua is often used in Singaporean slang to descri
   4   train      normal             none                     ah kua  In some linguistic studies, the term ah kua is analyzed for 
   5   train      normal             none                     ah kua  The term ah kua is a Hokkien loanword used in Singapore and 
   6   train   off

## Manual judge: single example
Pick a row by index from the table above, send it to the judge, and inspect the full result. Good for testing before the full run.

In [7]:
# ---- Change this index to judge any row from the table above ----
ROW_INDEX = 0

row = all_rows[ROW_INDEX]
term = row["source_term"]
entry = TERM_LOOKUP.get(term.casefold(), {})
context = extract_context(entry)
prompt = build_judge_prompt(row["text"], term, context)

print(f"Row {ROW_INDEX} | split={row['split']} | term={term}")
print(f"Assigned: label={row['label']}  target_group={row.get('target_group', 'none')}")
print(f"Text: {row['text']}")
print(f"\n{'='*80}\nCalling Gemini judge...\n{'='*80}\n")

raw = _call_gemini_json(prompt)
parsed = parse_judge_response(raw)

if parsed:
    agree = row["label"] == parsed["judge_label"]
    print(f"Judge label:        {parsed['judge_label']}  {'✓ AGREE' if agree else '✗ DISAGREE'}")
    print(f"Judge target_group: {parsed['judge_target_group']}")
    print(f"Judge confidence:   {parsed['judge_confidence']}")
    print(f"Singlish realism:   {parsed['singlish_realism']}")
    print(f"Reasoning:          {parsed['reasoning']}")
else:
    print("❌ Failed to parse judge response")
    print(f"Raw response: {raw[:500] if raw else '(empty)'}")

Row 0 | split=train | term=ah kua
Assigned: label=hate  target_group=chinese
Text: Chinese Singaporeans who are ah kua are a complete embarrassment to our race and deserve to be treated with nothing but contempt.

Calling Gemini judge...

Judge label:        hate  ✓ AGREE
Judge target_group: chinese
Judge confidence:   high
Singlish realism:   medium
Reasoning:          The text expresses explicit hatred and calls for contempt against Chinese Singaporeans based on their perceived sexual orientation or gender identity using the derogatory term 'ah kua'.


## Run judge pipeline
Iterates over all rows, calls Gemini for each, and writes results to a JSONL file.

**Resume support**: If `JUDGE_OUTPUT_FILE` already exists, completed rows are skipped on re-run. Set `RESET_JUDGE_OUTPUT = True` to start fresh.

In [8]:
RESET_JUDGE_OUTPUT = False

if RESET_JUDGE_OUTPUT and JUDGE_OUTPUT_FILE.exists():
    JUDGE_OUTPUT_FILE.unlink()
    print("Deleted previous judge output:", JUDGE_OUTPUT_FILE)


def _load_completed_indices(output_file):
    """Load set of (split, row_index) already judged for resume support."""
    completed = set()
    if not Path(output_file).exists():
        return completed
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                completed.add((row.get("split", ""), row.get("row_index", -1)))
            except json.JSONDecodeError:
                pass
    return completed


def run_judge(rows, output_file):
    """Run the Gemini judge on all rows, writing results incrementally."""
    completed = _load_completed_indices(output_file)
    if completed:
        print(f"📋 Resume mode: {len(completed)} rows already judged")

    total = len(rows)
    n_written = 0
    n_skipped = 0
    n_failed = 0
    start_time = time.time()

    with open(output_file, "a", encoding="utf-8") as fh:
        for idx, row in enumerate(rows):
            key = (row["split"], idx)
            if key in completed:
                n_skipped += 1
                continue

            term = row["source_term"]
            entry = TERM_LOOKUP.get(term.casefold(), {})
            context = extract_context(entry)
            prompt = build_judge_prompt(row["text"], term, context)

            raw = _call_gemini_json(prompt)
            parsed = parse_judge_response(raw)

            if parsed:
                result = {
                    "split": row["split"],
                    "row_index": idx,
                    "source_term": term,
                    "text": row["text"],
                    "assigned_label": row["label"],
                    "assigned_target_group": row.get("target_group", "none"),
                    "judge_label": parsed["judge_label"],
                    "judge_target_group": parsed["judge_target_group"],
                    "judge_confidence": parsed["judge_confidence"],
                    "singlish_realism": parsed["singlish_realism"],
                    "reasoning": parsed["reasoning"],
                    "agreement": row["label"] == parsed["judge_label"],
                }
                fh.write(json.dumps(result, ensure_ascii=False) + "\n")
                fh.flush()
                n_written += 1
                status = "✓" if result["agreement"] else "✗"
            else:
                n_failed += 1
                status = "⚠"

            done = n_written + n_skipped + n_failed
            if done % 10 == 0 or done == total:
                elapsed = time.time() - start_time
                rate = (n_written + n_failed) / elapsed if elapsed > 0 else 0
                print(
                    f"  [{done}/{total}] written={n_written} skipped={n_skipped} "
                    f"failed={n_failed} ({rate:.1f} rows/s) {status}",
                    flush=True,
                )

            time.sleep(0.15)  # Rate limiting

    elapsed = time.time() - start_time
    print(f"\n✅ Done in {elapsed:.1f}s")
    print(f"   Written: {n_written} | Skipped (resume): {n_skipped} | Failed: {n_failed}")
    print(f"   Output: {output_file}")


print(f"Ready to judge {len(all_rows)} rows → {JUDGE_OUTPUT_FILE}")
print(f"Resume: {'ON (existing file found)' if JUDGE_OUTPUT_FILE.exists() else 'OFF (fresh run)'}")

Ready to judge 540 rows → /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/judge_results.jsonl
Resume: OFF (fresh run)


In [9]:
# Run the judge
run_judge(all_rows, JUDGE_OUTPUT_FILE)

  [10/540] written=10 skipped=0 failed=0 (0.2 rows/s) ✓
  [20/540] written=20 skipped=0 failed=0 (0.2 rows/s) ✓
  [30/540] written=30 skipped=0 failed=0 (0.2 rows/s) ✓
  [40/540] written=40 skipped=0 failed=0 (0.2 rows/s) ✗
  [50/540] written=50 skipped=0 failed=0 (0.2 rows/s) ✗
  [60/540] written=60 skipped=0 failed=0 (0.2 rows/s) ✓
  [70/540] written=70 skipped=0 failed=0 (0.2 rows/s) ✓
  [80/540] written=80 skipped=0 failed=0 (0.2 rows/s) ✗
  [90/540] written=90 skipped=0 failed=0 (0.2 rows/s) ✓
  [100/540] written=100 skipped=0 failed=0 (0.2 rows/s) ✓
  [110/540] written=110 skipped=0 failed=0 (0.2 rows/s) ✓
  [120/540] written=120 skipped=0 failed=0 (0.2 rows/s) ✓
  [130/540] written=130 skipped=0 failed=0 (0.2 rows/s) ✓
  [140/540] written=140 skipped=0 failed=0 (0.2 rows/s) ✓
  [150/540] written=150 skipped=0 failed=0 (0.2 rows/s) ✓
  [160/540] written=160 skipped=0 failed=0 (0.2 rows/s) ✓
  [170/540] written=170 skipped=0 failed=0 (0.2 rows/s) ✓
  [180/540] written=180 skipped=

## Near-duplicate detection
Detect near-identical texts within each `(source_term, label)` group using `difflib.SequenceMatcher`. Flags pairs with similarity > 0.85.

In [10]:
SIMILARITY_THRESHOLD = 0.85

# Group all loaded rows by (term, label)
groups = defaultdict(list)
for row in all_rows:
    groups[(row["source_term"], row["label"])].append(row)

duplicates = []
exact_dupes = 0

for (term, label), rows_in_group in sorted(groups.items()):
    texts = [r["text"] for r in rows_in_group]
    seen_exact = set()
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            if texts[i] == texts[j]:
                key = (term, label, min(texts[i], texts[j]), max(texts[i], texts[j]))
                if key not in seen_exact:
                    seen_exact.add(key)
                    exact_dupes += 1
                    duplicates.append({
                        "term": term, "label": label,
                        "text_a": texts[i][:120], "text_b": texts[j][:120],
                        "similarity": 1.0, "type": "exact",
                    })
            else:
                ratio = difflib.SequenceMatcher(None, texts[i], texts[j]).ratio()
                if ratio >= SIMILARITY_THRESHOLD:
                    duplicates.append({
                        "term": term, "label": label,
                        "text_a": texts[i][:120], "text_b": texts[j][:120],
                        "similarity": round(ratio, 3), "type": "near",
                    })

print(f"Duplicate detection (threshold={SIMILARITY_THRESHOLD}):")
print(f"  Exact duplicates: {exact_dupes}")
print(f"  Near-duplicates: {len(duplicates) - exact_dupes}")
print(f"  Total flagged pairs: {len(duplicates)}")

if duplicates:
    print(f"\nTop flagged pairs (showing up to 15):")
    for d in sorted(duplicates, key=lambda x: -x["similarity"])[:15]:
        print(f"\n  [{d['type']}] {d['term']} / {d['label']} (sim={d['similarity']})")
        print(f"    A: {d['text_a']}")
        print(f"    B: {d['text_b']}")

Duplicate detection (threshold=0.85):
  Exact duplicates: 54
  Near-duplicates: 178
  Total flagged pairs: 232

Top flagged pairs (showing up to 15):

  [exact] CB kia / hate (sim=1.0)
    A: Those Malay Singaporeans are such a bunch of CB kia, they always cause trouble in our neighborhood.
    B: Those Malay Singaporeans are such a bunch of CB kia, they always cause trouble in our neighborhood.

  [exact] CB kia / offensive (sim=1.0)
    A: That CB kia just cut me off in traffic without even signaling!
    B: That CB kia just cut me off in traffic without even signaling!

  [exact] CB kia / offensive (sim=1.0)
    A: That CB kia driver just cut me off without signaling, what a jerk.
    B: That CB kia driver just cut me off without signaling, what a jerk.

  [exact] Zhu Bajie / normal (sim=1.0)
    A: In the classic novel Journey to the West, Zhu Bajie is a character known for his strength and appetite.
    B: In the classic novel Journey to the West, Zhu Bajie is a character known fo

## Similarity distribution by label
Inspect pairwise `difflib` similarity within each label separately, with extra focus on `hate` examples.

In [17]:
def _percentile(sorted_values, q):
    if not sorted_values:
        return 0.0
    if len(sorted_values) == 1:
        return float(sorted_values[0])
    pos = (len(sorted_values) - 1) * q
    lower = int(pos)
    upper = min(lower + 1, len(sorted_values) - 1)
    weight = pos - lower
    return sorted_values[lower] * (1 - weight) + sorted_values[upper] * weight


similarity_pairs_by_label = defaultdict(list)

for (term, label), rows_in_group in sorted(groups.items()):
    texts = [r["text"] for r in rows_in_group]
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            ratio = difflib.SequenceMatcher(None, texts[i], texts[j]).ratio()
            similarity_pairs_by_label[label].append(
                {
                    "term": term,
                    "similarity": ratio,
                    "text_a": texts[i],
                    "text_b": texts[j],
                }
            )

print("SIMILARITY DISTRIBUTION BY LABEL")
print("=" * 90)
for label in ["hate", "offensive", "normal"]:
    pairs = similarity_pairs_by_label.get(label, [])
    scores = sorted(p["similarity"] for p in pairs)
    if not scores:
        print(f"\n[{label}] No pairs found")
        continue

    over_70 = sum(1 for s in scores if s >= 0.70)
    over_85 = sum(1 for s in scores if s >= 0.85)
    over_95 = sum(1 for s in scores if s >= 0.95)
    mean_score = sum(scores) / len(scores)

    print(f"\n[{label.upper()}]")
    print(f"  pairs:   {len(scores)}")
    print(f"  mean:    {mean_score:.3f}")
    print(f"  median:  {_percentile(scores, 0.50):.3f}")
    print(f"  p75:     {_percentile(scores, 0.75):.3f}")
    print(f"  p90:     {_percentile(scores, 0.90):.3f}")
    print(f"  p95:     {_percentile(scores, 0.95):.3f}")
    print(f"  max:     {scores[-1]:.3f}")
    print(f"  >=0.70:  {over_70} ({over_70/len(scores)*100:.1f}%)")
    print(f"  >=0.85:  {over_85} ({over_85/len(scores)*100:.1f}%)")
    print(f"  >=0.95:  {over_95} ({over_95/len(scores)*100:.1f}%)")

print("\n" + "=" * 90)
print("HATE-ONLY DETAIL")
print("=" * 90)

hate_pairs = sorted(similarity_pairs_by_label.get("hate", []), key=lambda x: -x["similarity"])
if not hate_pairs:
    print("No hate pairs found")
else:
    hate_scores = sorted(p["similarity"] for p in hate_pairs)
    bins = [
        (0.00, 0.30),
        (0.30, 0.50),
        (0.50, 0.70),
        (0.70, 0.85),
        (0.85, 0.95),
        (0.95, 1.01),
    ]

    print("Hate similarity histogram:")
    for lo, hi in bins:
        count = sum(1 for s in hate_scores if lo <= s < hi)
        pct = count / len(hate_scores) * 100 if hate_scores else 0
        bar = "#" * int(pct / 2)
        print(f"  [{lo:.2f}, {hi:.2f}): {count:>4} ({pct:>5.1f}%) {bar}")

    print("\nMost similar hate pairs (top 15):")
    for item in hate_pairs[:15]:
        print(f"\n  term={item['term']} | sim={item['similarity']:.3f}")
        print(f"    A: {item['text_a'][:160]}")
        print(f"    B: {item['text_b'][:160]}")

SIMILARITY DISTRIBUTION BY LABEL

[HATE]
  pairs:   720
  mean:    0.596
  median:  0.585
  p75:     0.711
  p90:     0.825
  p95:     0.894
  max:     1.000
  >=0.70:  198 (27.5%)
  >=0.85:  60 (8.3%)
  >=0.95:  23 (3.2%)

[OFFENSIVE]
  pairs:   720
  mean:    0.694
  median:  0.689
  p75:     0.983
  p90:     1.000
  p95:     1.000
  max:     1.000
  >=0.70:  354 (49.2%)
  >=0.85:  228 (31.7%)
  >=0.95:  186 (25.8%)

[NORMAL]
  pairs:   720
  mean:    0.601
  median:  0.550
  p75:     0.768
  p90:     0.958
  p95:     1.000
  max:     1.000
  >=0.70:  235 (32.6%)
  >=0.85:  137 (19.0%)
  >=0.95:  80 (11.1%)

HATE-ONLY DETAIL
Hate similarity histogram:
  [0.00, 0.30):   14 (  1.9%) 
  [0.30, 0.50):  207 ( 28.7%) ##############
  [0.50, 0.70):  301 ( 41.8%) ####################
  [0.70, 0.85):  138 ( 19.2%) #########
  [0.85, 0.95):   37 (  5.1%) ##
  [0.95, 1.01):   23 (  3.2%) #

Most similar hate pairs (top 15):

  term=CB kia | sim=1.000
    A: Those Malay Singaporeans are such a b

## Analysis: Agreement metrics and confusion matrix
Load the judge results and compute agreement metrics.

In [11]:
# Load judge results
judge_rows = []
with open(JUDGE_OUTPUT_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            judge_rows.append(json.loads(line))

print(f"Loaded {len(judge_rows)} judge results from {JUDGE_OUTPUT_FILE}")

# ---------- Overall agreement ----------
n_agree = sum(1 for r in judge_rows if r["agreement"])
n_total = len(judge_rows)
agreement_pct = n_agree / n_total * 100 if n_total else 0
print(f"\n{'='*60}")
print(f"OVERALL AGREEMENT: {n_agree}/{n_total} ({agreement_pct:.1f}%)")
print(f"{'='*60}")

# ---------- Confusion matrix ----------
LABELS = ["hate", "offensive", "normal"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}
confusion = [[0] * 3 for _ in range(3)]
for r in judge_rows:
    a = label_to_idx.get(r["assigned_label"])
    j = label_to_idx.get(r["judge_label"])
    if a is not None and j is not None:
        confusion[a][j] += 1

print(f"\nConfusion Matrix (rows=assigned, cols=judge)")
header = f"{'':>12}" + "".join(f"{l:>12}" for l in LABELS) + f"{'total':>10}"
print(header)
print("-" * len(header))
for i, label in enumerate(LABELS):
    row_total = sum(confusion[i])
    row_str = f"{label:>12}" + "".join(f"{confusion[i][j]:>12}" for j in range(3)) + f"{row_total:>10}"
    print(row_str)
col_totals = [sum(confusion[i][j] for i in range(3)) for j in range(3)]
print(f"{'total':>12}" + "".join(f"{ct:>12}" for ct in col_totals) + f"{sum(col_totals):>10}")

# ---------- Per-label precision / recall ----------
print(f"\nPer-label metrics (treating judge as ground truth):")
print(f"{'label':>12} {'precision':>10} {'recall':>10} {'f1':>10} {'support':>10}")
print("-" * 54)
for i, label in enumerate(LABELS):
    tp = confusion[i][i]
    # Precision: of all judge predicted as this label, how many were assigned this label
    pred_total = col_totals[i]
    precision = tp / pred_total if pred_total else 0
    # Recall: of all assigned this label, how many did judge agree
    assigned_total = sum(confusion[i])
    recall = tp / assigned_total if assigned_total else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    print(f"{label:>12} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f} {assigned_total:>10}")

# ---------- Cohen's kappa ----------
pe = sum(
    (sum(confusion[i]) / n_total) * (col_totals[i] / n_total)
    for i in range(3)
) if n_total else 0
po = n_agree / n_total if n_total else 0
kappa = (po - pe) / (1 - pe) if (1 - pe) != 0 else 0
print(f"\nCohen's kappa: {kappa:.3f}")
print(f"  (po={po:.3f}, pe={pe:.3f})")

# ---------- Per-split agreement ----------
print(f"\nPer-split agreement:")
by_split = defaultdict(lambda: [0, 0])
for r in judge_rows:
    by_split[r["split"]][1] += 1
    if r["agreement"]:
        by_split[r["split"]][0] += 1
for split_name in sorted(by_split):
    agree, total = by_split[split_name]
    print(f"  {split_name}: {agree}/{total} ({agree/total*100:.1f}%)")

# ---------- Per-term agreement ----------
print(f"\nPer-term agreement:")
by_term = defaultdict(lambda: [0, 0])
for r in judge_rows:
    by_term[r["source_term"]][1] += 1
    if r["agreement"]:
        by_term[r["source_term"]][0] += 1
for term in sorted(by_term):
    agree, total = by_term[term]
    pct = agree / total * 100 if total else 0
    indicator = " ⚠" if pct < 70 else ""
    print(f"  {term:>30}: {agree}/{total} ({pct:.1f}%){indicator}")

Loaded 540 judge results from /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/judge_results.jsonl

OVERALL AGREEMENT: 462/540 (85.6%)

Confusion Matrix (rows=assigned, cols=judge)
                    hate   offensive      normal     total
----------------------------------------------------------
        hate         176           0           4       180
   offensive           7         136          37       180
      normal           0          30         150       180
       total         183         166         191       540

Per-label metrics (treating judge as ground truth):
       label  precision     recall         f1    support
------------------------------------------------------
        hate      0.962      0.978      0.970        180
   offensive      0.819      0.756      0.786        180
      normal      0.785      0.833      0.809        180

Cohen's kappa: 0.783
  (po=0.856, pe=0.333)

Per-split agreement:
  test: 154/180 (85.6%)
 

## Target group validation
For hate-labeled rows, check if the judge's target group matches the assigned one.

In [12]:
hate_rows = [r for r in judge_rows if r["assigned_label"] == "hate"]
n_hate = len(hate_rows)
n_tg_match = sum(
    1 for r in hate_rows if r["assigned_target_group"] == r["judge_target_group"]
)
print(f"Target group agreement (hate rows only):")
print(f"  {n_tg_match}/{n_hate} ({n_tg_match/n_hate*100:.1f}%)" if n_hate else "  No hate rows")

# Show mismatches
tg_mismatches = [r for r in hate_rows if r["assigned_target_group"] != r["judge_target_group"]]
if tg_mismatches:
    print(f"\nTarget group mismatches (showing up to 15):")
    for r in tg_mismatches[:15]:
        print(f"  term={r['source_term']:<25} assigned_tg={r['assigned_target_group']:<15} "
              f"judge_tg={r['judge_target_group']:<15}")
        print(f"    text: {r['text'][:120]}")
else:
    print("\n✅ All hate rows have matching target groups")

Target group agreement (hate rows only):
  169/180 (93.9%)

Target group mismatches (showing up to 15):
  term=apu neh neh               assigned_tg=indian          judge_tg=none           
    text: The term apu neh neh is a hateful and derogatory slur used to target Indian Singaporeans, reducing their identity to a r
  term=cina babi                 assigned_tg=malay           judge_tg=malay, chinese 
    text: Malay Singaporeans are just as disgusting as any cina babi and should leave.
  term=cina babi                 assigned_tg=indian          judge_tg=indian, chinese
    text: Indian Singaporeans are just as bad as those cina babi, they all should leave the country.
  term=keling                    assigned_tg=malay           judge_tg=malay, indian  
    text: Malay Singaporeans are just as worthless as any keling, they should all leave our country.
  term=cina babi                 assigned_tg=indian          judge_tg=indian, chinese
    text: Those Indian Singaporeans are just a

## Singlish realism summary
Distribution of Singlish realism scores across all judged rows.

In [13]:
realism_counts = Counter(r["singlish_realism"] for r in judge_rows)
print("Singlish realism distribution:")
for level in ["high", "medium", "low"]:
    count = realism_counts.get(level, 0)
    pct = count / len(judge_rows) * 100 if judge_rows else 0
    bar = "█" * int(pct / 2)
    print(f"  {level:>8}: {count:>4} ({pct:>5.1f}%) {bar}")

# Flag low-realism examples
low_realism = [r for r in judge_rows if r["singlish_realism"] == "low"]
if low_realism:
    print(f"\n⚠ Low-realism examples ({len(low_realism)} total, showing up to 10):")
    for r in low_realism[:10]:
        print(f"  [{r['assigned_label']}] {r['source_term']}: {r['text'][:100]}")
        print(f"    → {r['reasoning'][:120]}")
else:
    print("\n✅ No low-realism examples")

Singlish realism distribution:
      high:  382 ( 70.7%) ███████████████████████████████████
    medium:  136 ( 25.2%) ████████████
       low:   22 (  4.1%) ██

⚠ Low-realism examples (22 total, showing up to 10):
  [normal] ah kua: In some linguistic studies, the term ah kua is analyzed for its historical usage in regional dialect
    → The text uses the term in a neutral, academic, and descriptive context, discussing its linguistic history rather than us
  [normal] CB kia: The term CB kia is often used in casual Singlish to describe someone who is being very annoying.
    → The text is a neutral and educational description of the term's usage in Singlish, without any offensive or hateful inte
  [normal] cina babi: The term cina babi is a derogatory expression used to insult people of Chinese descent.
    → The text provides a neutral, descriptive definition of the term 'cina babi' without using it to express hatred or target
  [normal] lan jiao bin: The linguistic study discussed th

## Flagged disagreements
All rows where the judge label differs from the assigned label, sorted by confidence.

In [14]:
disagreements = [r for r in judge_rows if not r["agreement"]]
confidence_order = {"high": 0, "medium": 1, "low": 2}
disagreements.sort(key=lambda r: confidence_order.get(r["judge_confidence"], 3))

print(f"Total disagreements: {len(disagreements)} / {len(judge_rows)} "
      f"({len(disagreements)/len(judge_rows)*100:.1f}%)\n")

# Disagreement breakdown
print("Disagreement breakdown (assigned → judge):")
transition_counts = Counter(
    (r["assigned_label"], r["judge_label"]) for r in disagreements
)
for (a, j), count in sorted(transition_counts.items(), key=lambda x: -x[1]):
    print(f"  {a:>10} → {j:<10}: {count}")

print(f"\nFlagged disagreements (showing up to 25, sorted by judge confidence):")
print("=" * 100)
for i, r in enumerate(disagreements[:25], 1):
    print(f"\n[{i}] {r['source_term']} | assigned={r['assigned_label']} → judge={r['judge_label']} "
          f"| confidence={r['judge_confidence']} | realism={r['singlish_realism']}")
    print(f"  text: {r['text'][:200]}")
    print(f"  reasoning: {r['reasoning'][:200]}")

Total disagreements: 78 / 540 (14.4%)

Disagreement breakdown (assigned → judge):
   offensive → normal    : 37
      normal → offensive : 30
   offensive → hate      : 7
        hate → normal    : 4

Flagged disagreements (showing up to 25, sorted by judge confidence):

[1] anneh | assigned=offensive → judge=hate | confidence=high | realism=medium
  text: Don't be so anneh and keep acting like a complete idiot in front of everyone.
  reasoning: The text uses 'anneh' (a term for Indian men) as a derogatory adjective, equating the identity with being a 'complete idiot', which constitutes hate speech targeting an ethnic group.

[2] anneh | assigned=offensive → judge=hate | confidence=high | realism=high
  text: Stop acting like an anneh, it's really disgusting.
  reasoning: The text uses the term 'anneh' (a colloquial term for Indian men) in a derogatory context, labeling the identity or behavior associated with the group as 'disgusting,' which constitutes hate speech.

[3] anneh | assig

## Export summary CSV
Write all judge results to a CSV file for further analysis or sharing.

In [15]:
CSV_COLUMNS = [
    "split", "source_term", "text",
    "assigned_label", "judge_label",
    "assigned_target_group", "judge_target_group",
    "judge_confidence", "singlish_realism",
    "reasoning", "agreement",
]

with open(JUDGE_CSV_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    for r in judge_rows:
        writer.writerow(r)

print(f"✅ Exported {len(judge_rows)} rows to {JUDGE_CSV_FILE}")
print(f"   Columns: {CSV_COLUMNS}")

✅ Exported 540 rows to /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/judge_results.csv
   Columns: ['split', 'source_term', 'text', 'assigned_label', 'judge_label', 'assigned_target_group', 'judge_target_group', 'judge_confidence', 'singlish_realism', 'reasoning', 'agreement']


## Red flag analysis
Consolidated summary of potential issues found by the judge.

In [16]:
print("=" * 70)
print("RED FLAG ANALYSIS — CONSOLIDATED SUMMARY")
print("=" * 70)

# --- 1. Overall quality ---
print(f"\n1. OVERALL AGREEMENT: {n_agree}/{n_total} ({agreement_pct:.1f}%)")
print(f"   Cohen's kappa: {kappa:.3f}")
if agreement_pct >= 85:
    print("   → ✅ Strong agreement — data quality looks good")
elif agreement_pct >= 70:
    print("   → ⚠️ Moderate agreement — some labels may need review")
else:
    print("   → 🚩 Low agreement — significant labeling issues")

# --- 2. Confusion matrix highlights ---
print(f"\n2. CONFUSION MATRIX HIGHLIGHTS:")
print(f"   Hate correctly identified:      {confusion[0][0]}/{sum(confusion[0])} ({confusion[0][0]/sum(confusion[0])*100:.0f}%)")
print(f"   Offensive correctly identified:  {confusion[1][1]}/{sum(confusion[1])} ({confusion[1][1]/sum(confusion[1])*100:.0f}%)")
print(f"   Normal correctly identified:     {confusion[2][2]}/{sum(confusion[2])} ({confusion[2][2]/sum(confusion[2])*100:.0f}%)")

# Biggest off-diagonal issues
off_diag = []
for i in range(3):
    for j in range(3):
        if i != j and confusion[i][j] > 0:
            off_diag.append((LABELS[i], LABELS[j], confusion[i][j]))
off_diag.sort(key=lambda x: -x[2])
if off_diag:
    print(f"\n   Biggest misclassification flows:")
    for assigned, judged, count in off_diag[:5]:
        print(f"   → {assigned} assigned as {judged}: {count} rows")

# --- 3. Problem terms ---
print(f"\n3. PROBLEM TERMS (agreement < 70%):")
problem_terms = []
for t in sorted(by_term):
    a, n = by_term[t]
    pct = a / n * 100 if n else 0
    if pct < 70:
        problem_terms.append((t, a, n, pct))
if problem_terms:
    for t, a, n, pct in problem_terms:
        print(f"   🚩 {t}: {a}/{n} ({pct:.1f}%)")
        # Show what kind of disagreements
        term_dis = [r for r in disagreements if r["source_term"] == t]
        flows = Counter((r["assigned_label"], r["judge_label"]) for r in term_dis)
        for (al, jl), c in flows.most_common(3):
            print(f"      {al} → {jl}: {c}")
else:
    print("   ✅ No terms below 70% agreement")

# --- 4. Singlish realism concerns ---
n_low = realism_counts.get("low", 0)
n_med = realism_counts.get("medium", 0)
n_high = realism_counts.get("high", 0)
print(f"\n4. SINGLISH REALISM:")
print(f"   High: {n_high} ({n_high/n_total*100:.1f}%) | Medium: {n_med} ({n_med/n_total*100:.1f}%) | Low: {n_low} ({n_low/n_total*100:.1f}%)")
if n_low > n_total * 0.1:
    print(f"   🚩 {n_low} rows ({n_low/n_total*100:.1f}%) rated low realism — synthetic data sounds unnatural")
elif n_low > 0:
    print(f"   ⚠️ {n_low} rows rated low realism")
else:
    print("   ✅ No low-realism rows")

# --- 5. Near-duplicates ---
print(f"\n5. NEAR-DUPLICATES:")
print(f"   Exact duplicates: {exact_dupes}")
print(f"   Near-duplicates (>{SIMILARITY_THRESHOLD}): {len(duplicates) - exact_dupes}")
if exact_dupes > 0:
    print(f"   🚩 {exact_dupes} exact duplicate pairs — text diversity issue")
if len(duplicates) - exact_dupes > n_total * 0.05:
    print(f"   🚩 High near-duplicate rate — generation may be too repetitive")

# --- 6. Target group validation (hate rows) ---
print(f"\n6. TARGET GROUP VALIDATION (hate rows):")
print(f"   Agreement: {n_tg_match}/{n_hate} ({n_tg_match/n_hate*100:.1f}%)" if n_hate else "   No hate rows")
if n_hate and (n_hate - n_tg_match) > 0:
    n_tg_mis = n_hate - n_tg_match
    print(f"   ⚠️ {n_tg_mis} hate rows have mismatched target groups")

# --- 7. High-confidence disagreements (most concerning) ---
hc_dis = [r for r in disagreements if r["judge_confidence"] == "high"]
print(f"\n7. HIGH-CONFIDENCE DISAGREEMENTS: {len(hc_dis)}")
if hc_dis:
    print(f"   🚩 These are the most concerning — judge is confident the label is wrong:")
    for r in hc_dis[:10]:
        print(f"\n   [{r['source_term']}] assigned={r['assigned_label']} → judge={r['judge_label']}")
        print(f"   text: {r['text'][:150]}")
        print(f"   reason: {r['reasoning'][:150]}")
else:
    print("   ✅ No high-confidence disagreements")

print(f"\n{'=' * 70}")
print("END OF ANALYSIS")
print(f"{'=' * 70}")

RED FLAG ANALYSIS — CONSOLIDATED SUMMARY

1. OVERALL AGREEMENT: 462/540 (85.6%)
   Cohen's kappa: 0.783
   → ✅ Strong agreement — data quality looks good

2. CONFUSION MATRIX HIGHLIGHTS:
   Hate correctly identified:      176/180 (98%)
   Offensive correctly identified:  136/180 (76%)
   Normal correctly identified:     150/180 (83%)

   Biggest misclassification flows:
   → offensive assigned as normal: 37 rows
   → normal assigned as offensive: 30 rows
   → offensive assigned as hate: 7 rows
   → hate assigned as normal: 4 rows

3. PROBLEM TERMS (agreement < 70%):
   🚩 apu neh neh: 14/27 (51.9%)
      offensive → normal: 9
      hate → normal: 4
   🚩 big fuck: 18/27 (66.7%)
      normal → offensive: 9
   🚩 bo lan pa: 18/27 (66.7%)
      normal → offensive: 9
   🚩 cina babi: 18/27 (66.7%)
      offensive → normal: 9
   🚩 keling: 18/27 (66.7%)
      offensive → normal: 9

4. SINGLISH REALISM:
   High: 382 (70.7%) | Medium: 136 (25.2%) | Low: 22 (4.1%)
   ⚠️ 22 rows rated low realism

5